# OSCD auto-improvement and component ablations (Phase D2)

Run the fixed 12-trial search with leave-one-city-out validation on the 14 **training** cities. Every trial is checkpointed to `benchmarks/results/auto_improve_log.json`, including failures. Selection uses pooled validation F1 only.

This notebook **never reads the official test split** and does not score it. Test scoring is a separate one-shot protocol in `docs/paper/PREREGISTRATION.md` (Phase D2.6) and must only run after this log and the chosen systems are frozen.

Expected Drive layout is the same as `notebooks/oscd_benchmark.ipynb`: `My Drive/oscd/archive (1).zip` (images and training labels). Test labels are not used here.

Results are **unverified** until the owner pastes the log output into the project.

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/oscd'
REPO_URL = 'https://github.com/Aravind-Subramanian-16/satellite-change-detection.git'
REPO_DIR = '/content/repo'
OSCD_DIR = '/content/oscd'
SEED = 42
RUN_CNN_FINETUNE = False

IMAGES_ROOT = f'{OSCD_DIR}/images/Onera Satellite Change Detection dataset - Images'
TRAIN_LABELS_ROOT = f'{OSCD_DIR}/train_labels/Onera Satellite Change Detection dataset - Train Labels'
RESULT_JSON = f'{REPO_DIR}/benchmarks/results/auto_improve_log.json'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
archive = Path(DRIVE_DIR) / 'archive (1).zip'
if not archive.is_file():
    raise FileNotFoundError(f'Missing OSCD archive: {archive}')

!mkdir -p /content/oscd
!unzip -q -o "$archive" -d /content/oscd

if Path(REPO_DIR).is_dir():
    %cd /content/repo
    !git pull --ff-only
else:
    !git clone $REPO_URL $REPO_DIR
    %cd /content/repo

!python -m pip -q install -r requirements-benchmark.txt
!python -m pytest -q tests/test_auto_improve.py tests/test_radiometric_compositing.py tests/test_uncertainty.py

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f'{REPO_DIR}/benchmarks')
from learned_features import OFFICIAL_TEST_CITIES, assert_no_test_cities
import oscd_eval

images = Path(IMAGES_ROOT)
train_cities = oscd_eval.read_city_list(images / 'train.txt')
if len(train_cities) != 14:
    raise ValueError(f'Expected 14 training cities, found {len(train_cities)}')
assert_no_test_cities(train_cities)
if set(train_cities) & set(OFFICIAL_TEST_CITIES):
    raise ValueError('Training split contains an official test city')
labels_root = Path(TRAIN_LABELS_ROOT)
for city in train_cities:
    for directory in (images / city / 'imgs_1_rect', images / city / 'imgs_2_rect'):
        if not directory.is_dir():
            raise FileNotFoundError(f'{city}: missing acquisition at {directory}')
    if not (labels_root / city / 'cm' / f'{city}-cm.tif').is_file():
        raise FileNotFoundError(f'{city}: missing training label')
print(f'Verified {len(train_cities)} TRAIN cities; test cities are not read.')

The search is intentionally expensive: 12 systems × 14 leave-one-city-out folds, with threshold grids refit on the 13 fit cities in each fold. Do not add candidates or change the budget after seeing any result. The JSON is checkpointed after every trial, so an interrupted runtime leaves a partial log with `complete: false`.

In [ ]:
%cd /content/repo
!python benchmarks/auto_improve.py \
  --images-root "$IMAGES_ROOT" \
  --train-labels-root "$TRAIN_LABELS_ROOT" \
  --output benchmarks/results/auto_improve_log.json \
  --seed $SEED

Path(DRIVE_DIR, 'results').mkdir(parents=True, exist_ok=True)
!cp benchmarks/results/auto_improve_log.json benchmarks/results/auto_improve_log.md "$DRIVE_DIR/results/"

In [ ]:
import json

payload = json.loads(Path(RESULT_JSON).read_text())
if payload['metadata']['test_split_read'] is not False:
    raise ValueError('Log must never read the test split')
if payload['metadata']['complete'] is not True:
    raise ValueError('Log is incomplete; do not use it for selection')
if payload['metadata']['trials_attempted'] != 12:
    raise ValueError('Expected all 12 pre-registered trials')
print('SELECTION:', json.dumps(payload['selection'], indent=2))
print('RECOMMENDATION:', json.dumps(payload['recommendation'], indent=2))
print('ABLATIONS:', json.dumps(payload['ablations'], indent=2))
failed = [trial['system'] for trial in payload['trials'] if trial['status'] != 'ok']
print('Failed trials:', failed or 'none')

## Optional CNN fine-tuning

Set `RUN_CNN_FINETUNE = True` to fine-tune a **new** checkpoint on the 11 fit TRAIN cities and validate the threshold on the 3 held-out TRAIN cities. This does not read or modify `best_model.pt`/`model.onnx`. It is off by default because it requires a GPU runtime and is not part of the 12-trial spectral search.

In [ ]:
if RUN_CNN_FINETUNE:
    %cd /content/repo
    !python benchmarks/finetune_cnn.py \
      --images-root "$IMAGES_ROOT" \
      --train-labels-root "$TRAIN_LABELS_ROOT" \
      --output "$DRIVE_DIR/artifacts/cnn_finetuned_oscd.pt" \
      --seed $SEED
else:
    print('CNN fine-tuning skipped; spectral search log is the only output of this notebook.')

## Next steps

- Commit the complete `auto_improve_log.json`/`.md` from Drive, plus the learned artifact's metrics JSON (not the joblib) once trained.
- Record the selected system, final training-city parameters, artifact hashes and fusion configuration in the project before test access.
- Do **not** score the test split in this notebook. The separate v3 protocol in `docs/paper/PREREGISTRATION.md` permits one scoring event only, after the frozen configuration is committed.
- Report per-city variance and any negative trial results exactly as logged.